# HF market power on the bond-day panel

Builds a holder-weighted measure of hedge fund market power over dealers and merges it onto the canonical bond-day panel, exactly mirroring how `holder_dir` is built in `empirics_v1`.

Steps.
- pull the repo book at the fund and dealer grain, the only place we see both counterparties
- per fund and prior month, build fund-level power, predetermined so it does not look ahead
- holder-weight it to one value per bond and day, the analogue of `holder_dir`
- merge onto `monetary_policy_induced_position.csv`, untraded bonds stay missing and become the no-HF baseline

The notebook writes four holder-weighted measures. `holder_power` is the raw number of counterparties a fund finances through. `holder_power_hhi` is the HHI of a fund's financing across its dealers, the inverse concentration view, so its amplification sign should come out opposite. `holder_power_eff` is the effective number of dealers (`1 / HHI`), a size-robust counterparty count. `holder_power_opt` is the size-purged count, the residual of the count on log fund size, which isolates market power from fund size and is the measure for the clean test. All lagged one month. Runs in the ECB environment, not executed locally. Requires `empirics_v1` to have been re-run with `isin` kept as a string.

In [10]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN', autocommit=True)
cursor = cnxn.cursor()

# Bonds, for the security universe

In [11]:
df = pd.read_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\bond_timeseries_v2.csv')
df['Dates'] = pd.to_datetime(df['Dates'])
df['bond_maturity'] = pd.to_datetime(df['bond_maturity'])
df['residual_bond_maturity'] = (df['bond_maturity'] - df['Dates']).dt.days / 365
df = df[df['residual_bond_maturity'] >= 0]
df['collateral_country'] = df['ISIN'].str[:2]
df = df[df['collateral_country'].isin(['DE', 'IT'])]
df = df[df['PX_ASK'].notna()]
df = df.drop_duplicates(subset=['ISIN', 'Dates'], keep='first')

securities = tuple(df['ISIN'].unique())
len(securities)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_18816\1498429172.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['bond_maturity'] = pd.to_datetime(df['bond_maturity'])


746

# Dealer universe

The counterparties that face the Cayman investment funds. Restricting both legs to this set keeps the dealer identity clean.

In [12]:
query_dealers = f'''
SELECT DISTINCT dealer_id FROM (
    SELECT s.lender_id AS dealer_id
    FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
    LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower
        ON s.borrower_id = s_borrower.id
    WHERE s.business_date BETWEEN DATE '2021-01-04' AND DATE '2025-11-01'
      AND s.nominal_ccy = 'EUR'
      AND s.central_clearing = 'non-cleared'
      AND s.borrower_country_residence = 'KY'
      AND s_borrower.sector = 'IF'
      AND s.security_isin IN {securities}
    UNION
    SELECT s.borrower_id AS dealer_id
    FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
    LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender
        ON s.lender_id = s_lender.id
    WHERE s.business_date BETWEEN DATE '2021-01-04' AND DATE '2025-11-01'
      AND s.nominal_ccy = 'EUR'
      AND s.central_clearing = 'non-cleared'
      AND s.lender_country_residence = 'KY'
      AND s_lender.sector = 'IF'
      AND s.security_isin IN {securities}
  )
'''
df_dealers = pd.read_sql_query(query_dealers, cnxn)
dealer_ids = tuple(df_dealers['dealer_id'].unique())
print(f"Number of dealers facing HFs: {len(dealer_ids)}")

C:\Users\hermesf\AppData\Local\Temp\ipykernel_18816\277365912.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dealers = pd.read_sql_query(query_dealers, cnxn)


Number of dealers facing HFs: 18


# Repo at the fund and dealer grain

Volume only. Haircuts are unreliable and not used.

In [13]:
# LONG leg, HF borrows cash and posts the bond. fund = borrower, dealer = lender.
query_long = f'''
SELECT s.business_date,
       s.security_isin AS isin,
       s.borrower_id   AS fund_id,
       s.lender_id     AS dealer_id,
       SUM(s.nominal_euro) AS long_vol
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower
    ON s.borrower_id = s_borrower.id
WHERE s.business_date BETWEEN DATE '2021-01-04' AND DATE '2025-11-01'
  AND s.nominal_ccy = 'EUR'
  AND s.central_clearing = 'non-cleared'
  AND s.borrower_country_residence = 'KY'
  AND s_borrower.sector = 'IF'
  AND s.lender_id IN {dealer_ids}
  AND s.gnlcoll = 'SPEC'
  AND s.security_isin IN {securities}
GROUP BY s.business_date, s.security_isin, s.borrower_id, s.lender_id
'''
df_long = pd.read_sql_query(query_long, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_18816\3021428952.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_long = pd.read_sql_query(query_long, cnxn)


In [14]:
# SHORT leg, HF lends cash and receives the bond. fund = lender, dealer = borrower.
query_short = f'''
SELECT s.business_date,
       s.security_isin AS isin,
       s.lender_id     AS fund_id,
       s.borrower_id   AS dealer_id,
       SUM(s.nominal_euro) AS short_vol
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender
    ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN DATE '2021-01-04' AND DATE '2025-11-01'
  AND s.nominal_ccy = 'EUR'
  AND s.central_clearing = 'non-cleared'
  AND s.lender_country_residence = 'KY'
  AND s_lender.sector = 'IF'
  AND s.borrower_id IN {dealer_ids}
  AND s.gnlcoll = 'SPEC'
  AND s.security_isin IN {securities}
GROUP BY s.business_date, s.security_isin, s.lender_id, s.borrower_id
'''
df_short = pd.read_sql_query(query_short, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_18816\2100334216.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_short = pd.read_sql_query(query_short, cnxn)


In [15]:
dfb = df_long.merge(df_short, on=['business_date', 'isin', 'fund_id', 'dealer_id'], how='outer')
dfb[['long_vol', 'short_vol']] = dfb[['long_vol', 'short_vol']].fillna(0)
dfb['gross_vol'] = (dfb['long_vol'] + dfb['short_vol']) / 1e9
dfb['business_date'] = pd.to_datetime(dfb['business_date'])
print(dfb.shape)

(3007282, 7)


# Fund market power, prior month, predetermined

Four measures, all lagged one month.
- `fund_n_dealers`, the number of distinct dealers a fund finances through. Higher means more outside options and more power. Raw headline, but mechanically rising in fund size.
- `fund_hhi`, the Herfindahl of the fund's financing across those dealers. Higher means more concentrated and less power, so it is the mirror of the count and its amplification sign should be opposite.
- `fund_eff_dealers`, the effective number of dealers (`1 / fund_hhi`). Discounts trivial relationships and is scale-free in volume, so it keeps the counterparty-count reading with far less size conflation. Monotone transform of the HHI.
- `fund_n_dealers_opt`, the size-purged count: the residual of `fund_n_dealers` on log `fund_book` (pooled OLS). Strips the mechanical "bigger fund, more dealers" channel, so downstream buckets compare connectedness at given size rather than small vs large funds. This is the measure that isolates market power from size.

In [ ]:
dfb['ym'] = dfb['business_date'].dt.to_period('M')

# fund-dealer monthly book
fd_m = dfb.groupby(['ym', 'fund_id', 'dealer_id'])['gross_vol'].sum().reset_index()

# the fund's weight on each of its dealers, for the HHI
fd_m['fund_book'] = fd_m.groupby(['ym', 'fund_id'])['gross_vol'].transform('sum')
fd_m['w_fd'] = fd_m['gross_vol'] / fd_m['fund_book']

fund_power = (fd_m.groupby(['ym', 'fund_id'])
                .agg(fund_n_dealers=('dealer_id', 'nunique'),
                     fund_hhi=('w_fd', lambda s: (s ** 2).sum()),
                     fund_book=('gross_vol', 'sum'))
                .reset_index())

# effective number of dealers, the numbers-equivalent of the HHI (inverse Herfindahl).
# discounts trivial relationships: a 90/9/1/.../. split counts as ~1, not as the raw
# dealer count. bounded in [1, fund_n_dealers] and scale-free in volume, so it keeps the
# counterparty-count reading with far less mechanical size conflation than the raw count.
fund_power['fund_eff_dealers'] = 1.0 / fund_power['fund_hhi']

# size-purged dealer count: residual of the count on log fund size (pooled OLS over
# fund-months). isolates whether a fund finances through more dealers than its size
# predicts, so the downstream buckets compare connectedness at given size rather than
# small vs large funds. positive = more connected than its size implies (genuine outside
# options), negative = unusually dependent for its size. this is the size-neutral measure.
_x = np.log(fund_power['fund_book'])
_m = _x.notna() & fund_power['fund_n_dealers'].notna()
_b, _a = np.polyfit(_x[_m], fund_power.loc[_m, 'fund_n_dealers'], 1)
fund_power['fund_n_dealers_opt'] = fund_power['fund_n_dealers'] - (_a + _b * _x)

# lag one month so power is predetermined relative to the outcome
fund_power['ym'] = fund_power['ym'] + 1

# Holder-weighted power at the bond-day grain

The analogue of `holder_dir`. For each bond and day, a gross-position-weighted average of the holders' power. Funds whose power is undefined drop out of both numerator and denominator.

In [ ]:
# fund-bond-day holdings, the weights
fb = dfb.groupby(['business_date', 'isin', 'fund_id'])['gross_vol'].sum().reset_index()
fb['ym'] = fb['business_date'].dt.to_period('M')
fb = fb.merge(fund_power[['ym', 'fund_id', 'fund_n_dealers', 'fund_hhi',
                          'fund_eff_dealers', 'fund_n_dealers_opt']],
              on=['ym', 'fund_id'], how='left')

# raw count, HHI concentration, effective number of dealers (1/HHI),
# and the size-purged count (residual of the count on log fund size)
fb['num_cnt'] = (fb['gross_vol'] * fb['fund_n_dealers']).where(fb['fund_n_dealers'].notna(), 0.0)
fb['den_cnt'] = fb['gross_vol'].where(fb['fund_n_dealers'].notna(), 0.0)
fb['num_hhi'] = (fb['gross_vol'] * fb['fund_hhi']).where(fb['fund_hhi'].notna(), 0.0)
fb['den_hhi'] = fb['gross_vol'].where(fb['fund_hhi'].notna(), 0.0)
fb['num_eff'] = (fb['gross_vol'] * fb['fund_eff_dealers']).where(fb['fund_eff_dealers'].notna(), 0.0)
fb['den_eff'] = fb['gross_vol'].where(fb['fund_eff_dealers'].notna(), 0.0)
fb['num_opt'] = (fb['gross_vol'] * fb['fund_n_dealers_opt']).where(fb['fund_n_dealers_opt'].notna(), 0.0)
fb['den_opt'] = fb['gross_vol'].where(fb['fund_n_dealers_opt'].notna(), 0.0)

holder = (fb.groupby(['business_date', 'isin'])
            .agg(num_cnt=('num_cnt', 'sum'), den_cnt=('den_cnt', 'sum'),
                 num_hhi=('num_hhi', 'sum'), den_hhi=('den_hhi', 'sum'),
                 num_eff=('num_eff', 'sum'), den_eff=('den_eff', 'sum'),
                 num_opt=('num_opt', 'sum'), den_opt=('den_opt', 'sum'))
            .reset_index())
holder['holder_power']     = holder['num_cnt'] / holder['den_cnt'].where(holder['den_cnt'] > 0)
holder['holder_power_hhi'] = holder['num_hhi'] / holder['den_hhi'].where(holder['den_hhi'] > 0)
holder['holder_power_eff'] = holder['num_eff'] / holder['den_eff'].where(holder['den_eff'] > 0)
holder['holder_power_opt'] = holder['num_opt'] / holder['den_opt'].where(holder['den_opt'] > 0)
holder = holder[['business_date', 'isin', 'holder_power', 'holder_power_hhi',
                 'holder_power_eff', 'holder_power_opt']]
print(holder.shape)
holder.head()

# Merge onto the canonical bond-day panel

Left merge, so bonds with no HF holder stay missing and serve as the no-HF baseline, exactly as `holder_dir` does.

In [18]:
# the canonical panel must already carry isin as a STRING, see the empirics_v1 change
panel = pd.read_csv('monetary_policy_induced_position.csv')
panel = panel.drop(columns=[c for c in panel.columns if c.startswith('Unnamed')], errors='ignore')
panel['business_date'] = pd.to_datetime(panel['business_date'])
holder['business_date'] = pd.to_datetime(holder['business_date'])

panel = panel.merge(holder, on=['business_date', 'isin'], how='left')
print('rows:', len(panel),
      '| HF-held bond-days with power:', int(panel['holder_power'].notna().sum()))

panel.to_csv('monetary_policy_induced_position_power.csv', index=False)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_18816\3740372399.py:2: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  panel = pd.read_csv('monetary_policy_induced_position.csv')


rows: 478483 | HF-held bond-days with power: 244103


In [ ]:
# same holder measures merged onto the OVERNIGHT panel (positions filtered to
# contractual_maturity <= 1 in empirics_v1_overnight). the holder_* columns are
# computed from the full repo book and are identical here; only the bond-day panel
# they attach to changes. this feeds the overnight position-adjustment LP.
panel_on = pd.read_csv('monetary_policy_induced_position_overnight.csv')
panel_on = panel_on.drop(columns=[c for c in panel_on.columns if c.startswith('Unnamed')], errors='ignore')
panel_on['business_date'] = pd.to_datetime(panel_on['business_date'])

panel_on = panel_on.merge(holder, on=['business_date', 'isin'], how='left')
print('overnight rows:', len(panel_on),
      '| HF-held bond-days with power:', int(panel_on['holder_power'].notna().sum()))

panel_on.to_csv('monetary_policy_induced_position_overnight_power.csv', index=False)